# Two-dimensional Ising model: Metropolis, Wolff and the critical point

## 1. The model and what a phase transition is
The Ising model puts a spin, +1 or -1, on every site of a square lattice. Neighbouring spins that agree lower the energy, and neighbours that disagree raise it. At low temperature the spins mostly agree and the lattice has a net magnetization. At high temperature thermal motion scrambles them and the net magnetization averages to zero. In an infinite lattice the change between the two happens at one sharp temperature, the critical temperature `Tc = 2 / ln(1 + sqrt(2))`, about 2.2692 in units where `J = 1`. That sharp change is a phase transition.

I cannot simulate an infinite lattice. On a finite lattice the transition is smeared out and the size of the smearing depends on the lattice size L. Comparing lattices of different L is how I locate `Tc` and read off the critical exponents. The exact values `Tc`, `beta = 1/8`, `gamma = 7/4` and `nu = 1` are known from the exact solution, so this project checks my simulation against them.

This notebook reads only `results/`, `data/` and `config.py`. Every number shown is printed by a code cell.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
import config
import ising

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLS = {8: "#8c6bb1", 16: "#2a6fbb", 32: "#2a9d8f", 64: "#d9822b"}
TC = config.TC_EXACT
obs = pd.read_csv("results/observables.csv")
grid = obs[(obs.algorithm == "metropolis") & (obs.tag != "Tc")].copy()
tcrun = obs[obs.tag == "Tc"].copy()
cross = pd.read_csv("results/crossings.csv")
expo = pd.read_csv("results/exponents.csv")
dyn = pd.read_csv("results/dynamics.csv")
summ = pd.read_csv("results/summary.csv").set_index("key")["value"]
print("exact Tc =", round(TC, 6))
print("rows: grid runs", len(grid), "| Tc runs", len(tcrun), "| lattice sizes", config.L_LIST, "| temperatures in grid", len(config.T_GRID))
print("measured sweeps per Metropolis run:", config.N_MEAS, "| blocks:", config.N_BLOCKS)


exact Tc = 2.269185
rows: grid runs 100 | Tc runs 8 | lattice sizes [8, 16, 32, 64] | temperatures in grid 25
measured sweeps per Metropolis run: 500000 | blocks: 50


## 2. Energy and magnetization against temperature
Points are the simulations (error bars are jackknife errors and are mostly smaller than the markers). The black curves are the exact infinite-lattice results. Magnetization on a finite lattice is shown as the mean of `|M|` per site, because the sign of `M` flips between runs and averages to zero.

In [2]:
Ts = np.linspace(config.T_GRID[0], config.T_GRID[-1], 400)
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
for L in config.L_LIST:
    d = grid[grid.L == L].sort_values("T")
    axs[0].errorbar(d["T"], d["e_mean"], yerr=d["e_mean_err"], fmt="o-", ms=3, lw=0.8, color=COLS[L], label=f"L = {L}")
    axs[1].errorbar(d["T"], d["absm_mean"], yerr=d["absm_mean_err"], fmt="o-", ms=3, lw=0.8, color=COLS[L], label=f"L = {L}")
axs[0].plot(Ts, [ising.u_exact(t) for t in Ts], color="black", lw=1.5, label="exact, infinite lattice")
axs[1].plot(Ts, [ising.m0_exact(t) for t in Ts], color="black", lw=1.5, label="exact, infinite lattice")
for ax in axs:
    ax.axvline(TC, color="grey", ls=":", lw=1); ax.set_xlabel("temperature T")
axs[0].set_ylabel("energy per site"); axs[1].set_ylabel("mean |magnetization| per site")
axs[0].legend(frameon=False, fontsize=8); axs[1].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig("Images/energy_magnetization.png", dpi=150); plt.close(fig)
print("saved Images/energy_magnetization.png")
ex = pd.read_csv("results/exact_comparison.csv")
sel = ex[np.isclose(ex["T"], [2.0, 2.2, 2.27, 2.3, 2.4, 2.6][0]) | ex["T"].isin([2.0, 2.2, 2.3, 2.4, 2.6])]
print(sel.to_string(index=False))


saved Images/energy_magnetization.png
  T  L    e_mean  e_mean_err   u_exact  absm_mean  absm_mean_err  m0_exact
2.0 64 -1.745615    0.000064 -1.745565   0.911365       0.000045  0.911319
2.2 64 -1.546311    0.000236 -1.546489   0.785053       0.000520  0.784755
2.3 64 -1.355560    0.000919 -1.342866   0.435286       0.005483  0.000000
2.4 64 -1.204179    0.000200 -1.203974   0.157191       0.001213  0.000000
2.6 64 -1.028350    0.000099 -1.028294   0.074467       0.000267  0.000000


![energy and magnetization](Images/energy_magnetization.png)

**Reading the figure.** At low and high temperature the L = 64 points sit on the exact infinite-lattice curves. Near `Tc` the finite lattices round off the exact curves. The magnetization of finite lattices does not drop to zero at `Tc` and stays positive above it, because `|M|` per site is positive even for pure noise and shrinks only as L grows. The energy at T = 2.3 for L = 64 is below (more negative than) the exact value in the table, which is a finite-size effect of the same kind.

## 3. Specific heat and susceptibility
The specific heat per site is `C = (<E^2> - <E>^2) / (N T^2)`. The connected susceptibility is `chi_c = (<M^2> - <|M|>^2) / (N T)`. Both should peak near `Tc`, grow with L, and move toward `Tc` as L grows.

In [3]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
for L in config.L_LIST:
    d = grid[grid.L == L].sort_values("T")
    axs[0].errorbar(d["T"], d["C"], yerr=d["C_err"], fmt="o-", ms=3, lw=0.8, color=COLS[L], label=f"L = {L}")
    axs[1].errorbar(d["T"], d["chi_c"], yerr=d["chi_c_err"], fmt="o-", ms=3, lw=0.8, color=COLS[L], label=f"L = {L}")
for ax in axs:
    ax.axvline(TC, color="grey", ls=":", lw=1); ax.set_xlabel("temperature T"); ax.legend(frameon=False, fontsize=8)
axs[0].set_ylabel("specific heat per site"); axs[1].set_ylabel("connected susceptibility per site")
fig.tight_layout(); fig.savefig("Images/heat_susceptibility.png", dpi=150); plt.close(fig)
print("saved Images/heat_susceptibility.png")
for L in config.L_LIST:
    d = grid[grid.L == L]
    iC = d["C"].idxmax(); ic = d["chi_c"].idxmax()
    print(f"L = {L}: C peaks at T = {d.loc[iC, 'T']:.2f} (value {d.loc[iC, 'C']:.3f}); chi_c peaks at T = {d.loc[ic, 'T']:.2f} (value {d.loc[ic, 'chi_c']:.3f})")


saved Images/heat_susceptibility.png
L = 8: C peaks at T = 2.35 (value 1.191); chi_c peaks at T = 2.55 (value 1.756)
L = 16: C peaks at T = 2.33 (value 1.555); chi_c peaks at T = 2.40 (value 6.062)
L = 32: C peaks at T = 2.30 (value 1.909); chi_c peaks at T = 2.34 (value 20.769)
L = 64: C peaks at T = 2.29 (value 2.252); chi_c peaks at T = 2.29 (value 70.840)


![heat and susceptibility](Images/heat_susceptibility.png)

**Reading the figure.** The peaks of both quantities grow with L and move toward `Tc` as L grows. The susceptibility peak is at 2.55 for L = 8, at 2.40 for L = 16, 2.34 for L = 32 and 2.29 for L = 64, all above the exact `Tc`. The specific heat peak value grows from 1.191 to 2.252 over the four sizes, a slow rise that is consistent with the logarithmic divergence of the exact solution, though four sizes cannot show that by themselves. The grid step of 0.01 near the peak limits how precisely the peak positions are located.

## 4. Binder cumulant and the primary result
The Binder cumulant is `U = 1 - <M^4> / (3 <M^2>^2)`. Curves for different L cross near `Tc`. For each pair of sizes the crossing is the last place on the fine grid where `U_small - U_large` goes from non-positive to positive, with linear interpolation. Errors come from a block bootstrap over all runs involved (2000 replicates). The ranges are indicative intervals.

In [4]:
fine = grid[grid["T"].isin(config.T_FINE)]
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4))
for L in config.L_LIST:
    d = fine[fine.L == L].sort_values("T")
    axs[0].errorbar(d["T"], d["U"], yerr=d["U_err"], fmt="o-", ms=3, lw=0.8, color=COLS[L], label=f"L = {L}")
    axs[1].errorbar(d["T"], d["U"], yerr=d["U_err"], fmt="o-", ms=3, lw=0.8, color=COLS[L], label=f"L = {L}")
for ax in axs:
    ax.axvline(TC, color="black", ls="--", lw=1.2, label="exact Tc" if ax is axs[0] else None)
    ax.set_xlabel("temperature T"); ax.set_ylabel("Binder cumulant U")
pcol = {"8,16": "#8c6bb1", "16,32": "#2a6fbb", "32,64": "#d9822b"}
for _, c in cross.iterrows():
    axs[0].axvline(c["Tc_estimate"], color=pcol[c["pair"]], ls=":", lw=1.4, label=f"crossing {c['pair']}")
prim = cross[cross.pair == "32,64"].iloc[0]
axs[1].axvspan(prim["lo"], prim["hi"], color="#d9822b", alpha=0.18, label="indicative interval, 32 and 64")
axs[1].axvline(prim["Tc_estimate"], color="#d9822b", ls=":", lw=1.6)
axs[1].set_xlim(2.24, 2.30); axs[1].set_ylim(0.55, 0.64)
axs[0].legend(frameon=False, fontsize=7); axs[1].legend(frameon=False, fontsize=7)
axs[0].set_title("fine grid"); axs[1].set_title("zoom near the crossing of L = 32 and 64")
fig.tight_layout(); fig.savefig("Images/binder_crossing.png", dpi=150); plt.close(fig)
print("saved Images/binder_crossing.png")
show = cross.copy()
display(show)
print(show.to_string(index=False))
print()
print(f"PRIMARY: Tc from the crossing of L = 32 and 64 = {prim['Tc_estimate']:.4f}, bootstrap standard error {prim['se']:.4f}, indicative interval [{prim['lo']:.4f}, {prim['hi']:.4f}]")
print(f"exact Tc = {prim['Tc_exact']:.4f}; difference = {prim['diff']:+.4f} = {prim['diff_in_se']:+.2f} standard errors; interval covers the exact value: {bool(prim['interval_covers_exact'])}")
print("bootstrap replicates that gave NaN:", int(prim['n_nan']))


saved Images/binder_crossing.png


,pair,Tc_estimate,se,lo,hi,n_nan,Tc_exact,diff,diff_in_se,interval_covers_exact
0,"8,16",2.263383,0.001972,2.258503,2.266821,0,2.269185,-0.005803,-2.942821,False
1,"16,32",2.270492,0.002388,2.265107,2.273886,0,2.269185,0.001307,0.547283,True
2,"32,64",2.266560,0.003464,2.259402,2.272863,0,2.269185,-0.002625,-0.757716,True


 pair  Tc_estimate       se       lo       hi  n_nan  Tc_exact      diff  diff_in_se  interval_covers_exact
 8,16     2.263383 0.001972 2.258503 2.266821      0  2.269185 -0.005803   -2.942821                  False
16,32     2.270492 0.002388 2.265107 2.273886      0  2.269185  0.001307    0.547283                   True
32,64     2.266560 0.003464 2.259402 2.272863      0  2.269185 -0.002625   -0.757716                   True

PRIMARY: Tc from the crossing of L = 32 and 64 = 2.2666, bootstrap standard error 0.0035, indicative interval [2.2594, 2.2729]
exact Tc = 2.2692; difference = -0.0026 = -0.76 standard errors; interval covers the exact value: True
bootstrap replicates that gave NaN: 0


![Binder cumulant](Images/binder_crossing.png)

**Reading the figure.** The primary estimate is 2.2666 with a bootstrap standard error of 0.0035 and an indicative interval of [2.2594, 2.2729]. The exact value is 2.2692, the difference is -0.0026, which is -0.76 standard errors, and the interval covers the exact value. The pair (16, 32) also covers it. The pair (8, 16) gives 2.2634, which is -2.94 standard errors from the exact value and is not covered. The likely reason is corrections to scaling on the smallest lattices (a likely reason, not a proven one), which shift the crossing of small sizes. The crossings are linear interpolations on a grid with spacing 0.01, which adds a systematic error that the bootstrap does not include.

## 5. Finite-size scaling
At `Tc` the susceptibility grows as `L^(gamma/nu)` and the magnetization falls as `L^(-beta/nu)`. The derivative of `U` with respect to T grows as `L^(1/nu)`, estimated here from the difference of `U` at 2.26 and 2.28. Each is a straight line on a log-log plot. Solid lines are least-squares fits to sizes 16, 32 and 64 (primary). Dashed lines have the exact slope and pass through the L = 32 point.

In [5]:
def yvals(name):
    if name == "gamma/nu":
        d = tcrun[tcrun.algorithm == "metropolis"].sort_values("L"); return d["L"].to_numpy(), d["chi"].to_numpy(), d["chi_err"].to_numpy()
    if name == "beta/nu":
        d = tcrun[tcrun.algorithm == "metropolis"].sort_values("L"); return d["L"].to_numpy(), d["absm_mean"].to_numpy(), d["absm_mean_err"].to_numpy()
    Ls, ys = [], []
    for L in config.L_LIST:
        lo = grid[(grid.L == L) & np.isclose(grid["T"], config.SLOPE_T_LOW)].iloc[0]
        hi = grid[(grid.L == L) & np.isclose(grid["T"], config.SLOPE_T_HIGH)].iloc[0]
        Ls.append(L); ys.append((lo["U"] - hi["U"]) / (config.SLOPE_T_HIGH - config.SLOPE_T_LOW))
    return np.array(Ls), np.array(ys), np.full(len(Ls), np.nan)

fig, axs = plt.subplots(1, 3, figsize=(14, 4.2))
exact_slope = {"gamma/nu": config.EXACT_GAMMA_OVER_NU, "beta/nu": -config.EXACT_BETA_OVER_NU, "1/nu": config.EXACT_ONE_OVER_NU}
ylabel = {"gamma/nu": "susceptibility at Tc", "beta/nu": "mean |m| at Tc", "1/nu": "slope of U between 2.26 and 2.28"}
for ax, name in zip(axs, ["gamma/nu", "beta/nu", "1/nu"]):
    L, y, e = yvals(name)
    ax.errorbar(L, y, yerr=e, fmt="o", color="black", ms=4)
    m = L >= 16
    slope, icpt = np.polyfit(np.log(L[m]), np.log(y[m]), 1)
    xx = np.linspace(L.min() * 0.9, L.max() * 1.1, 50)
    ax.plot(xx, np.exp(icpt) * xx ** slope, color="#2a6fbb", lw=1.6, label=f"fit, slope {slope:.3f}")
    y32 = y[L == 32][0]
    ax.plot(xx, y32 * (xx / 32.0) ** exact_slope[name], color="#b03a2e", ls="--", lw=1.4, label=f"exact slope {exact_slope[name]:g}")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("L"); ax.set_ylabel(ylabel[name]); ax.set_title(name)
    ax.set_xticks([8, 16, 32, 64]); ax.set_xticklabels(["8", "16", "32", "64"]); ax.minorticks_off(); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig("Images/scaling_fits.png", dpi=150); plt.close(fig)
print("saved Images/scaling_fits.png")
print(expo.to_string(index=False))


saved Images/scaling_fits.png
quantity     fit      sizes  estimate       se       lo       hi  n_nan  exact      diff  diff_in_se
gamma/nu primary   16 32 64  1.742867 0.006448 1.730437 1.755163      0  1.750 -0.007133   -1.106263
gamma/nu     all 8 16 32 64  1.747374 0.003818 1.739780 1.754629      0  1.750 -0.002626   -0.687770
 beta/nu primary   16 32 64  0.129658 0.004478 0.121179 0.138360      0  0.125  0.004658    1.040196
 beta/nu     all 8 16 32 64  0.127043 0.002678 0.121990 0.132425      0  0.125  0.002043    0.762611
    1/nu primary   16 32 64  0.846407 0.118122 0.606355 1.070728      0  1.000 -0.153593   -1.300288
    1/nu     all 8 16 32 64  0.950071 0.071474 0.801482 1.087014      0  1.000 -0.049929   -0.698565


![scaling fits](Images/scaling_fits.png)

**Reading the figure.** For `gamma/nu` the primary fit gives 1.743 (exact 1.75) and the all-sizes fit gives 1.747. For `beta/nu` the primary fit gives 0.130 (exact 0.125) and the all-sizes fit gives 0.127. Every difference from the exact value is within 1.11 standard errors. The `1/nu` fit is much noisier because it uses a difference of two Binder cumulants. The primary fit gives 0.846 with a standard error of 0.118 and the all-sizes fit gives 0.950 with 0.071. Both intervals contain 1, but they are wide. The bootstrap errors come from one seed per run and do not include corrections to scaling, so agreement within these errors does not rule out small systematic offsets.

## 6. Critical slowing down
The integrated autocorrelation time of `|M|` at `Tc`, in sweeps. For Wolff the time in cluster updates is converted to sweep equivalents by multiplying with the mean cluster size over N. The slope on log axes is the dynamic exponent z. With four lattice sizes this is a rough estimate.

In [6]:
per = dyn[dyn.row_type == "per_L"]
zfit = dyn[dyn.row_type == "z_fit"].set_index("algorithm")
fig, ax = plt.subplots(figsize=(6.2, 4.4))
for alg, col, lab in [("metropolis", "#d9822b", "Metropolis (sweeps)"), ("wolff", "#2a6fbb", "Wolff (sweep equivalents)")]:
    d = per[per.algorithm == alg].sort_values("L")
    ax.plot(d["L"], d["tau_absm_sweeps"], "o", color=col, label=f"{lab}, z = {zfit.loc[alg, 'z']:.2f}")
    xx = np.linspace(7, 70, 50)
    s, ic = np.polyfit(np.log(d["L"]), np.log(d["tau_absm_sweeps"]), 1)
    ax.plot(xx, np.exp(ic) * xx ** s, color=col, lw=1)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("L"); ax.set_ylabel("integrated autocorrelation time of |M|")
ax.set_xticks([8, 16, 32, 64]); ax.set_xticklabels(["8", "16", "32", "64"]); ax.minorticks_off(); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig("Images/critical_slowing_down.png", dpi=150); plt.close(fig)
print("saved Images/critical_slowing_down.png")
print(per.to_string(index=False))
print(zfit[["z", "z_se"]].to_string())
for L in config.L_LIST:
    print(f"L = {L}: Metropolis tau / Wolff tau (both in sweep equivalents) = {float(summ[f'speedup_L{L}']):.1f}")
w = per[per.algorithm == "wolff"]
print("Wolff tau of |M| in cluster updates:", dict(zip(w["L"], w["tau_absm"].round(3))))


saved Images/critical_slowing_down.png
row_type  algorithm    L   tau_absm     tau_E  mean_cluster_fraction  tau_absm_sweeps   z  z_se
   per_L metropolis  8.0   3.135449  2.374352                    NaN         3.135449 NaN   NaN
   per_L      wolff  8.0   1.542184  1.690912               0.645487         0.995459 NaN   NaN
   per_L metropolis 16.0  10.402545  5.631642                    NaN        10.402545 NaN   NaN
   per_L      wolff 16.0   2.206437  2.573463               0.547320         1.207627 NaN   NaN
   per_L metropolis 32.0  41.675066 16.473661                    NaN        41.675066 NaN   NaN
   per_L      wolff 32.0   2.991160  3.798900               0.456109         1.364294 NaN   NaN
   per_L metropolis 64.0 184.362943 52.656541                    NaN       184.362943 NaN   NaN
   per_L      wolff 64.0   3.884892  5.390263               0.389357         1.512609 NaN   NaN
                   z      z_se
algorithm                     
metropolis  1.963545  0.066264
wolf

![critical slowing down](Images/critical_slowing_down.png)

**Reading the figure.** The Metropolis autocorrelation time of `|M|` grows from 3.1 sweeps at L = 8 to 184.4 sweeps at L = 64, and the fitted dynamic exponent is z = 1.96 with a least-squares standard error of 0.07. The value usually quoted for single-spin Metropolis is about 2.17, so my estimate is lower. Four small lattices and an integrated time of `|M|` only, with no correction for the smallest size, make this a rough estimate, and the quoted standard error only reflects the scatter around a straight line. The Wolff time in sweep equivalents grows from 1.0 to 1.5 and gives z = 0.20 with a standard error of 0.02. At L = 64 the Metropolis time is about 121.9 times the Wolff time in sweep equivalents. This compares decorrelation per sweep equivalent. It does not compare the computing time, because a Wolff update in my pure Python code costs more per spin than a numpy sweep.

## 7. Are the error bars honest
A run is flagged `blocks_short` when the block length is below 10 autocorrelation times. The jackknife error of such a run is probably too small. The second table is the 4 x 4 exact check: for a lattice with 65536 states every estimator can be computed exactly, and the pull is the simulated value minus the exact value divided by the jackknife error.

In [7]:
met = obs[obs.algorithm == "metropolis"]
print("Metropolis runs flagged blocks_short:", int(met.blocks_short.sum()), "of", len(met))
print(met.groupby("L").blocks_short.agg(["sum", "count"]).rename(columns={"sum": "flagged", "count": "runs"}).to_string())
fl = met[met.blocks_short][["L", "T", "tau_absm", "block_length_over_tau_absm"]].sort_values(["L", "T"])
print(fl.to_string(index=False))
print()
print("Wolff runs flagged:", int(obs[obs.algorithm == "wolff"].blocks_short.sum()), "of", int((obs.algorithm == "wolff").sum()))
ec = pd.read_csv("results/exact_check_4x4.csv")
print(ec.to_string(index=False))
print("largest absolute pull:", round(ec["pull"].abs().max(), 2))
print("number of pulls:", len(ec), "| pulls with |pull| > 2:", int((ec["pull"].abs() > 2).sum()))


Metropolis runs flagged blocks_short: 0 of 104
    flagged  runs
L                
8         0    26
16        0    26
32        0    26
64        0    26
Empty DataFrame
Columns: [L, T, tau_absm, block_length_over_tau_absm]
Index: []

Wolff runs flagged: 0 of 4
 algorithm        T  quantity  simulated  jackknife_se     exact      pull
metropolis 2.000000    e_mean  -1.754992      0.001017 -1.755380  0.381407
metropolis 2.000000 absm_mean   0.918894      0.000436  0.918943 -0.112865
metropolis 2.000000         C   0.605485      0.003124  0.605533 -0.015160
metropolis 2.000000     chi_c   0.195330      0.001846  0.195720 -0.211160
metropolis 2.000000         U   0.644837      0.000163  0.644811  0.158185
     wolff 2.000000    e_mean  -1.755360      0.001297 -1.755380  0.015649
     wolff 2.000000 absm_mean   0.918863      0.000504  0.918943 -0.160251
metropolis 2.269185    e_mean  -1.563369      0.001197 -1.565624  1.883606
metropolis 2.269185 absm_mean   0.842950      0.000580  0.8438

**Reading the tables.** No Metropolis or Wolff run is flagged `blocks_short`. The longest autocorrelation time at `Tc` is 184.4 sweeps against blocks of 10000 sweeps. The flag only tests block length against the autocorrelation time of `|M|`, and the jackknife errors still come from one seed per run, so it does not guarantee that the error bars are exact. In the 4 x 4 exact check the largest absolute pull is 2.60 (Wolff, `absm_mean` at T = 3.0), only one of the 21 pulls exceeds 2, and none is near the limit of 5 that would indicate a wrong sampler.

## 8. What this shows and what it does not

**What this shows.**
- The Binder crossing of L = 32 and 64 gives a critical temperature of 2.2666 with an indicative interval of [2.2594, 2.2729], which covers the exact 2.2692. The difference is -0.76 standard errors.
- `gamma/nu` and `beta/nu` come out within 1.11 standard errors of 1.75 and 0.125 for both choices of sizes. `1/nu` is consistent with 1 but with wide intervals.
- Metropolis gives z = 1.96 with a standard error of 0.07 from four sizes. Wolff gives z = 0.20 with 0.02. At L = 64 the Wolff update decorrelates about 121.9 times faster than Metropolis per sweep equivalent.
- The sampler agrees with exact enumeration on 4 x 4 lattices at three temperatures.

**What this does not show.**
- The small-lattice pair (8, 16) misses the exact `Tc` by 2.94 standard errors, so corrections to scaling are visible. A likely reason, not a proven one, is the small size.
- Four lattice sizes are not enough to fit corrections. The `1/nu` result in particular is weakly constrained.
- The error bars come from one seed per run and the crossing is a linear interpolation on a grid with spacing 0.01. Neither is part of the quoted bootstrap errors.
- The z values are rough. My Metropolis value is below the commonly quoted 2.17, and I did not find out why.
- Nothing here says anything about three dimensions or about lattices larger than 64.